# 🛡️ SmartAPD - PPE Detection Training

**Cara Pakai:**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Tunggu ~30-60 menit
4. Download model

In [ ]:
# Check GPU
!nvidia-smi -L

In [ ]:
# Install (FIXED VERSION - compatible with Colab)
!pip install ultralytics==8.0.200 kagglehub pyyaml -qq

import ultralytics
print(f"✅ Ultralytics: {ultralytics.__version__}")

In [ ]:
# Download dataset from Kaggle
import kagglehub
import os

path = kagglehub.dataset_download("snehilsanyal/construction-site-safety-image-dataset-roboflow")
print(f"Downloaded: {path}")

# Find css-data folder
dataset_location = None
for root, dirs, files in os.walk(path):
    if 'train' in dirs:
        dataset_location = root
        break
if not dataset_location:
    dataset_location = os.path.join(path, 'css-data')

print(f"\n✅ Dataset: {dataset_location}")
!ls {dataset_location}

In [ ]:
# Create data.yaml
import yaml

CLASSES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
           'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']

with open('data.yaml', 'w') as f:
    yaml.dump({
        'train': f'{dataset_location}/train',
        'val': f'{dataset_location}/valid',
        'test': f'{dataset_location}/test',
        'nc': 10,
        'names': CLASSES
    }, f)

print("✅ data.yaml created")
!cat data.yaml

In [ ]:
%%time
# Train YOLOv8
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

model = YOLO('yolov8s.pt')

results = model.train(
    data='data.yaml',
    epochs=80,
    batch=16,
    imgsz=640,
    patience=25,
    name='smartapd_ppe',
    exist_ok=True,
    device=0,
    verbose=True
)

print("\n✅ Training complete!")

In [ ]:
# Save model
import shutil
import os

best = 'runs/detect/smartapd_ppe/weights/best.pt'
output = '/content/smartapd_ppe_model.pt'

if os.path.exists(best):
    shutil.copy(best, output)
    size = os.path.getsize(output) / (1024*1024)
    print(f"✅ Model saved: {output}")
    print(f"📦 Size: {size:.2f} MB")
else:
    print("❌ Model not found - check training output")

In [ ]:
# Download model to your computer
from google.colab import files
files.download('/content/smartapd_ppe_model.pt')

## ✅ Done!

Pindahkan `smartapd_ppe_model.pt` ke:
```
d:\PROJECT PROJECT KU\smartapd\ai-engine\smartapd_ppe_model.pt
```